In [12]:
from utils import *

In [13]:
r = 3
g = 3
d = 1 # no tocar
X = Curve("X", g)
J = Jacobian(X).to_lambda()

In [14]:
obj = motive_generic_clean(X, r, d)

In [15]:
monoms = get_small_monomials(X, r)
max_dim = (r**2-1)*(g-1)
coefs = get_coefficients(max_dim)

In [16]:
def find_motive_dfs(obj: LambdaRingExpr, X: Curve, coefs: list[LambdaRingExpr], monoms, n_rounds: int) -> list[LambdaRingExpr]:
    """
    returns, if found, a motivic decompostion given monomials (high degree) and coefficients
    """
    max_degree = max(monoms.keys())
    min_degree = min(monoms.keys())
    n_monoms = len(monoms[max_degree])
    n_coefs = len(coefs)
    state0 = {
        "degree": max_degree,
        "monom_idx": 0,
        "round": 0,
        "coef_idx": 0
    }
    to_explore = [(obj, 0, state0)]
    while to_explore:
        remaining, big_part, state = to_explore[-1]
        candidate = find_motive_low(remaining, X)
        if candidate is not None:
            return [candidate + big_part]
        monom_X, monom_H = monoms[state["degree"]][state["monom_idx"]]
        coef = coefs[state["coef_idx"]]
        m = (remaining - coef*monom_H).expand()
        new_state = {
            "degree": state["degree"],
            "monom_idx": state["monom_idx"],
            "round": state["round"] + 1,       # COMO HAGO ESTOOOOOOOOOOO
            "coef_idx": state["coef_idx"]
        }
        if new_state["coef_idx"] == n_coefs:
            new_state["coef_idx"] = 0
            new_state["round"] += 1
            to_explore.pop()
        if new_state["round"] == n_rounds:
            new_state["round"] = 0
            new_state["monom_idx"] += 1
        if new_state["monom_idx"] == n_monoms:
            new_state["monom_idx"] = 0
            new_state["degree"] -= 1 # asumimos que no hay saltos de degree
        if new_state["degree"] >= min_degree and not any(term.could_extract_minus_sign() for term in m.as_ordered_terms()):
            to_explore.append((m, coef*monom_X, new_state))
        n_monoms = len(monoms[new_state["degree"]])
    return []

In [17]:
find_motive_bfs(obj, X, coefs, monoms, 2).pop()

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:02<00:00, 11.07it/s]


L**16 + L**14*λ1(X) + L**13*λ1(X) + L**12*λ2(X) + L**11*λ1(X)**2 + L**10*λ2(X) + L**10*λ3(X) + L**9*λ1(X)*λ2(X) + L**8*λ1(X)*λ2(X) + L**8*λ4(X) + L**7*λ1(X)*λ3(X) + L**7*λ3(X) + L**6*λ2(X)**2 + L**6*λ3(X) + L**5*λ1(X)*λ2(X) + L**5*λ1(X)*λ3(X) + L**4*λ1(X)*λ2(X) + L**4*λ2(X) + L**4*λ4(X) + L**3*λ1(X)**2 + L**3*λ3(X) + L**2*λ1(X) + L**2*λ2(X) + L*λ1(X) + 1